In [3]:
import torch
import torch.nn as nn 
import torch.nn.functional as f 
import numpy as np

"""Regular ConvNN Attention Implementation"""
class MultiHeadConvNNAttention(nn.Module):
    def __init__(self, 
                 d_hidden,
                 num_heads, 
                 attention_dropout, 
                 K, 
                 convolution_type='depthwise',
                 seq_length=197):

        super(MultiHeadConvNNAttention, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"

        self.d_hidden = d_hidden 
        self.num_heads = num_heads 
        self.attention_dropout = attention_dropout 
        self.d_k = d_hidden // num_heads 
        self.K = K 
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        self.in_channels = d_hidden // num_heads
        self.out_channels = d_hidden // num_heads

        if convolution_type == 'standard': 
            self.conv = nn.Conv1d(
                in_channels=self.in_channels,
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                bias=False
            )
        elif convolution_type == 'depthwise':
            self.conv = nn.Conv1d(
                in_channels=self.in_channels, 
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                groups=self.in_channels, 
                bias=False
            )
        elif convolution_type == 'depthwise-separable':
            self.conv = nn.Sequential(
                # Depthwise Convolution
                nn.Conv1d(
                    in_channels=self.in_channels,
                    out_channels=self.in_channels,
                    kernel_size=self.K,
                    stride=self.K,
                    padding=0,
                    groups=self.in_channels,
                    bias=False
                ), 
                # Pointwise Convolution
                nn.Conv1d(
                    in_channels=self.in_channels,
                    out_channels=self.out_channels,
                    kernel_size=1,
                    stride=1,
                    padding=0, 
                    bias=False
                )
            )
        self.conv.weight.data.fill_(1.0)

    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size() 
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2) # (B, num_heads, seq_length, d_k)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size() 
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden)

    def _prime(self, v, qk, K):
        # v: (B*num_heads, d_k, seq_length), qk: (B*num_heads, seq_length, seq_length)
        b, c, t = v.shape
        topk_values, topk_indices = torch.topk(qk, k=K, dim=2, largest=True)

        topk_values = torch.softmax(topk_values, dim=-1)
        topk_indices_exp = topk_indices.unsqueeze(1).expand(b, c, t, K)
        topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K)

        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
        prime = topk_values_exp * prime
        prime = prime.view(b, c, -1)
        return prime

    def forward(self, x):
        B = x.shape[0]

        # Linear Projection + Split Heads 
        q = self.split_head(self.W_q(x)) # (B, NH, SL, DK)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        # Attention Matrix: (B, NH, SL, SL) - Q @ K^T
        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        # Merge B and num_heads into dim for prime & conv 
        ## (B, NH, SL, DK) → (B*NH, DK, SL) for v and (B, NH, SL, SL) → (B*NH, SL, SL)
        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k).permute(0, 2, 1)
        am_merged = attn_matrix.reshape(B * self.num_heads, self.seq_length, self.seq_length)

        # Prime and Convolution 
        prime = self._prime(v_merged, am_merged, self.K)
        out = self.conv(prime) # (B*num_heads, d_k, seq_length) 

        # Reshape back: (B*NH, DK, SL) → (B, NH, SL, DK)
        out = out.permute(0, 2, 1).contiguous().view(B, self.num_heads, self.seq_length, self.d_k)
        out = self.dropout(out) 

        # Combine Heads and Final Linear Projection
        output = self.W_o(self.combine_heads(out)) # (B, SL, d_hidden)
        return output
        

In [10]:
import torch
import torch.nn as nn
import numpy as np
import triton
import triton.language as tl

# ==========================================
# 1. TRITON KERNEL (FORWARD PASS)
# ==========================================
@triton.jit
def fused_prime_conv_fwd_kernel(
    # Pointers to matrices
    v_ptr, indices_ptr, values_ptr, weight_ptr, out_ptr,
    # Strides to handle memory layout
    stride_vb, stride_vt, stride_vd,
    stride_ib, stride_it, stride_ik,
    stride_wb, stride_wk,
    stride_ob, stride_ot, stride_od,
    # Matrix dimensions
    B_NH, T, D: tl.constexpr, K: tl.constexpr,
    # Meta-parameters
    BLOCK_D: tl.constexpr
):
    """
    Fuses the gathering of V, multiplication by Top-K attention weights, 
    and the depthwise convolution step.
    """
    pid_b_t = tl.program_id(0) # 1D grid covering Batch*Heads and Seq_len
    pid_b = pid_b_t // T
    pid_t = pid_b_t % T

    # Set up channel offsets
    d_offsets = tl.arange(0, BLOCK_D)
    mask_d = d_offsets < D

    # Initialize accumulator for the convolution sum
    acc = tl.zeros([BLOCK_D], dtype=tl.float32)

    # Loop over the Top-K elements
    for k in range(K):
        # 1. Load the index and attention value for the k-th top element
        idx_offset = pid_b * stride_ib + pid_t * stride_it + k * stride_ik
        v_idx = tl.load(indices_ptr + idx_offset)
        attn_val = tl.load(values_ptr + idx_offset)

        # 2. Load the V vector for all D channels at the gathered index
        v_offsets = pid_b * stride_vb + v_idx * stride_vt + d_offsets * stride_vd
        v_vec = tl.load(v_ptr + v_offsets, mask=mask_d, other=0.0)

        # 3. Load the Depthwise Convolution weight for this K step
        w_offsets = d_offsets * stride_wb + k * stride_wk
        w_vec = tl.load(weight_ptr + w_offsets, mask=mask_d, other=0.0)

        # 4. Multiply and accumulate (Gather * Attn_Value * Conv_Weight)
        acc += v_vec * attn_val * w_vec

    # Store the final convolved output
    out_offsets = pid_b * stride_ob + pid_t * stride_ot + d_offsets * stride_od
    tl.store(out_ptr + out_offsets, acc, mask=mask_d)


# ==========================================
# 2. AUTOGRAD WRAPPER (FORWARD + BACKWARD)
# ==========================================
class FusedPrimeConvFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, v, topk_indices, topk_values, conv_weight):
        # Save tensors needed for the backward pass
        ctx.save_for_backward(v, topk_indices, topk_values, conv_weight)
        
        B_NH, T, D = v.shape
        _, _, K = topk_indices.shape
        
        # Ensure contiguous memory for predictable strides
        v = v.contiguous()
        topk_indices = topk_indices.contiguous()
        topk_values = topk_values.contiguous()
        weight = conv_weight.squeeze(1).contiguous() # Squeeze from (D, 1, K) to (D, K)
        
        out = torch.empty_like(v)
        
        # Grid computes one block per query token per batch/head
        grid = lambda meta: (B_NH * T, )
        BLOCK_D = triton.next_power_of_2(D)
        
        fused_prime_conv_fwd_kernel[grid](
            v, topk_indices, topk_values, weight, out,
            v.stride(0), v.stride(1), v.stride(2),
            topk_indices.stride(0), topk_indices.stride(1), topk_indices.stride(2),
            weight.stride(0), weight.stride(1),
            out.stride(0), out.stride(1), out.stride(2),
            B_NH, T, D, K,
            BLOCK_D=BLOCK_D
        )
        return out

    @staticmethod
    def backward(ctx, grad_out):
        v, topk_indices, topk_values, conv_weight = ctx.saved_tensors
        B_NH, T, D = v.shape
        _, _, K = topk_indices.shape
        weight = conv_weight.squeeze(1) # (D, K)
        
        grad_out = grad_out.contiguous()

        # Flatten indices to (B_NH, T*K, 1) and expand to D channels
        idx_flat = topk_indices.view(B_NH, T * K, 1).expand(-1, -1, D)
        
        # Gather V directly to shape (B_NH, T*K, D), then reshape
        v_gathered = torch.gather(v, 1, idx_flat).view(B_NH, T, K, D)

        # Pre-compute shapes for broadcasting
        grad_out_exp = grad_out.unsqueeze(2)                # (B_NH, T, 1, D)
        val_exp = topk_values.unsqueeze(-1)                 # (B_NH, T, K, 1)
        weight_t_exp = weight.t().unsqueeze(0).unsqueeze(0) # (1, 1, K, D)

        # Gradient w.r.t topk_values (dVal)
        grad_val = (grad_out_exp * v_gathered * weight_t_exp).sum(dim=-1) # (B_NH, T, K)

        # Gradient w.r.t conv_weight (dW)
        grad_weight_raw = (grad_out_exp * v_gathered * val_exp).sum(dim=(0, 1)) # (K, D)
        grad_weight = grad_weight_raw.t().unsqueeze(1) # Transpose & unsqueeze back to (D, 1, K)

        # Gradient w.r.t V (dV)
        dv_gathered = grad_out_exp * val_exp * weight_t_exp # (B_NH, T, K, D)
        grad_v = torch.zeros_like(v)
        
        # Scatter add the gradients back to the original V locations
        grad_v.scatter_add_(1, idx_flat, dv_gathered.view(B_NH, T * K, D))

        # topk_indices is discrete, so its gradient is None.
        return grad_v, None, grad_val, grad_weight



######### NEW backward kernel code but resulted in slower speed ###########
# @triton.jit
# def fused_prime_conv_bwd_kernel(
#     # Pointers
#     grad_out_ptr, v_ptr, indices_ptr, values_ptr, weight_ptr,
#     grad_v_ptr, grad_val_ptr, grad_weight_ptr,
#     # Strides
#     stride_gob, stride_got, stride_god,
#     stride_vb, stride_vt, stride_vd,
#     stride_ib, stride_it, stride_ik,
#     stride_wb, stride_wk,
#     # Dimensions
#     B_NH, T, D: tl.constexpr, K: tl.constexpr,
#     BLOCK_D: tl.constexpr
# ):
#     """
#     Fuses the backward pass, calculating dV, dVal, and dW entirely in SRAM
#     and safely scattering the results back using atomic adds.
#     """
#     pid = tl.program_id(0)
#     pid_b = pid // T
#     pid_t = pid % T

#     d_offsets = tl.arange(0, BLOCK_D)
#     mask_d = d_offsets < D

#     # 1. Load the incoming gradient (dO) for this specific query token
#     go_offsets = pid_b * stride_gob + pid_t * stride_got + d_offsets * stride_god
#     go_vec = tl.load(grad_out_ptr + go_offsets, mask=mask_d, other=0.0)

#     # Loop over the Top-K elements
#     for k in range(K):
#         # Load the gathered index and the attention value
#         idx_offset = pid_b * stride_ib + pid_t * stride_it + k * stride_ik
#         idx = tl.load(indices_ptr + idx_offset)
#         val = tl.load(values_ptr + idx_offset)

#         # Load the original V vector at the gathered index
#         v_offsets = pid_b * stride_vb + idx * stride_vt + d_offsets * stride_vd
#         v_vec = tl.load(v_ptr + v_offsets, mask=mask_d, other=0.0)

#         # Load the Convolution Weight for this K step
#         w_offsets = d_offsets * stride_wb + k * stride_wk
#         w_vec = tl.load(weight_ptr + w_offsets, mask=mask_d, other=0.0)

#         # --------------------------------------------------
#         # GRADIENT CALCULATIONS & ATOMIC SCATTERS
#         # --------------------------------------------------

#         # 1. Gradient w.r.t Top-K Values (dVal)
#         # Math: sum_d(dO * V * W)
#         g_val_vec = go_vec * v_vec * w_vec
#         g_val = tl.sum(g_val_vec, axis=0)
#         tl.store(grad_val_ptr + idx_offset, g_val)

#         # 2. Gradient w.r.t V (dV)
#         # Math: dO * val * W 
#         # (Must use atomic_add because multiple queries might attend to the same V index)
#         g_v_update = go_vec * val * w_vec
#         tl.atomic_add(grad_v_ptr + v_offsets, g_v_update, mask=mask_d)

#         # 3. Gradient w.r.t Convolution Weights (dW)
#         # Math: dO * V * val
#         # (Must use atomic_add because all threads update the same global weights)
#         g_w_update = go_vec * v_vec * val
#         tl.atomic_add(grad_weight_ptr + w_offsets, g_w_update, mask=mask_d)

# class FusedPrimeConvFunction(torch.autograd.Function):
#     @staticmethod
#     def forward(ctx, v, topk_indices, topk_values, conv_weight):
#         ctx.save_for_backward(v, topk_indices, topk_values, conv_weight)
        
#         B_NH, T, D = v.shape
#         _, _, K = topk_indices.shape
        
#         v = v.contiguous()
#         topk_indices = topk_indices.contiguous()
#         topk_values = topk_values.contiguous()
#         weight = conv_weight.squeeze(1).contiguous()
        
#         out = torch.empty_like(v)
#         grid = lambda meta: (B_NH * T, )
#         BLOCK_D = triton.next_power_of_2(D)
        
#         fused_prime_conv_fwd_kernel[grid](
#             v, topk_indices, topk_values, weight, out,
#             v.stride(0), v.stride(1), v.stride(2),
#             topk_indices.stride(0), topk_indices.stride(1), topk_indices.stride(2),
#             weight.stride(0), weight.stride(1),
#             out.stride(0), out.stride(1), out.stride(2),
#             B_NH, T, D, K,
#             BLOCK_D=BLOCK_D
#         )
#         return out

#     @staticmethod
#     def backward(ctx, grad_out):
#         v, topk_indices, topk_values, conv_weight = ctx.saved_tensors
#         B_NH, T, D = v.shape
#         _, _, K = topk_indices.shape
#         weight = conv_weight.squeeze(1).contiguous()
        
#         grad_out = grad_out.contiguous()

#         # Initialize output gradient tensors
#         # grad_v and grad_weight MUST be zeroed out because we atomic_add into them
#         grad_v = torch.zeros_like(v)
#         grad_val = torch.empty_like(topk_values) # Doesn't need zeros, we do a direct store
#         grad_weight = torch.zeros_like(weight)
        
#         grid = lambda meta: (B_NH * T, )
#         BLOCK_D = triton.next_power_of_2(D)

#         fused_prime_conv_bwd_kernel[grid](
#             grad_out, v, topk_indices, topk_values, weight,
#             grad_v, grad_val, grad_weight,
#             grad_out.stride(0), grad_out.stride(1), grad_out.stride(2),
#             v.stride(0), v.stride(1), v.stride(2),
#             topk_indices.stride(0), topk_indices.stride(1), topk_indices.stride(2),
#             weight.stride(0), weight.stride(1),
#             B_NH, T, D, K,
#             BLOCK_D=BLOCK_D
#         )

#         # Reshape grad_weight back to PyTorch's expected Depthwise Conv1d shape: (D, 1, K)
#         return grad_v, None, grad_val, grad_weight.unsqueeze(1)


# ==========================================
# 3. PYTORCH MODULE
# ==========================================
class FastMultiHeadConvNNAttention(nn.Module):
    def __init__(self, 
                 d_hidden,
                 num_heads, 
                 attention_dropout, 
                 K, 
                 seq_length=197):

        super(FastMultiHeadConvNNAttention, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"

        self.d_hidden = d_hidden 
        self.num_heads = num_heads 
        self.attention_dropout = attention_dropout 
        self.d_k = d_hidden // num_heads 
        self.K = K 
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        # Depthwise Convolution weights matching PyTorch's native Conv1d shape
        self.conv_weight = nn.Parameter(torch.ones(self.d_k, 1, self.K))

    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size() 
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x):
        B = x.shape[0]

        # Linear Projection + Split Heads 
        q = self.split_head(self.W_q(x)) # (B, NH, SL, DK)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        # Attention Matrix: (B, NH, SL, SL)
        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        # Top-K Selection
        topk_values, topk_indices = torch.topk(attn_matrix, k=self.K, dim=-1, largest=True)
        topk_values = torch.softmax(topk_values, dim=-1)

        # Merge Batch and Heads for the Triton Kernel
        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k)
        topk_indices_merged = topk_indices.reshape(B * self.num_heads, self.seq_length, self.K)
        topk_values_merged = topk_values.reshape(B * self.num_heads, self.seq_length, self.K)

        # Apply Fused Triton Operation (Forward + Backward handled automatically)
        out = FusedPrimeConvFunction.apply(
            v_merged, topk_indices_merged, topk_values_merged, self.conv_weight
        )

        # Reshape back: (B*NH, SL, DK) → (B, NH, SL, DK) → (B, SL, d_hidden)
        out = out.view(B, self.num_heads, self.seq_length, self.d_k)
        out = out.transpose(1, 2).contiguous().view(B, self.seq_length, self.d_hidden)
        out = self.dropout(out) 

        # Final Linear Projection
        output = self.W_o(out)
        return output


# ==========================================
# 4. QUICK VERIFICATION TEST
# ==========================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    if device.type != "cuda":
        print("Warning: Triton requires a CUDA-enabled GPU. This test will fail on CPU.")
    else:
        # Hyperparameters matching standard ViT-Base
        BATCH_SIZE = 2
        SEQ_LENGTH = 197
        D_HIDDEN = 768
        NUM_HEADS = 12
        K = 8
        
        print("Initializing FastMultiHeadConvNNAttention...")
        model = FastMultiHeadConvNNAttention(
            d_hidden=D_HIDDEN, 
            num_heads=NUM_HEADS, 
            attention_dropout=0.1, 
            K=K, 
            seq_length=SEQ_LENGTH
        ).to(device)
        
        # Dummy input
        x = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device, requires_grad=True)
        
        print("Running Forward Pass...")
        out = model(x)
        print(f"Output shape: {out.shape} (Expected: [{BATCH_SIZE}, {SEQ_LENGTH}, {D_HIDDEN}])")
        
        print("Running Backward Pass...")
        loss = out.sum()
        loss.backward()
        print(f"Input gradient shape: {x.grad.shape}")
        print("Success! Forward and backward passes completed.")

Initializing FastMultiHeadConvNNAttention...
Running Forward Pass...
Output shape: torch.Size([2, 197, 768]) (Expected: [2, 197, 768])
Running Backward Pass...
Input gradient shape: torch.Size([2, 197, 768])
Success! Forward and backward passes completed.


In [11]:
import torch
import numpy as np

# (Assume MultiHeadConvNNAttention and FastMultiHeadConvNNAttention are defined above this)

def benchmark_module(module, x, num_iters=100):
    # 1. Warm-up
    # GPUs have initialization overhead. We run a few dummy passes first.
    for _ in range(10):
        out = module(x)
        loss = out.sum()
        loss.backward()
    
    torch.cuda.synchronize()
    
    # 2. Memory Benchmark
    # Forward Pass Memory
    torch.cuda.reset_peak_memory_stats()
    out = module(x)
    fwd_mem = torch.cuda.max_memory_allocated() / (1024 ** 2) # Convert to MB
    
    # Backward Pass Memory
    torch.cuda.reset_peak_memory_stats()
    loss = out.sum()
    loss.backward()
    bwd_mem = torch.cuda.max_memory_allocated() / (1024 ** 2) # Convert to MB
    
    # 3. Speed Benchmark using CUDA Events
    fwd_times = []
    bwd_times = []
    
    for _ in range(num_iters):
        # Time Forward
        torch.cuda.synchronize()
        start_fwd = torch.cuda.Event(enable_timing=True)
        end_fwd = torch.cuda.Event(enable_timing=True)
        
        start_fwd.record()
        out = module(x)
        end_fwd.record()
        torch.cuda.synchronize()
        fwd_times.append(start_fwd.elapsed_time(end_fwd))
        
        # Time Backward
        loss = out.sum()
        torch.cuda.synchronize()
        start_bwd = torch.cuda.Event(enable_timing=True)
        end_bwd = torch.cuda.Event(enable_timing=True)
        
        start_bwd.record()
        loss.backward()
        end_bwd.record()
        torch.cuda.synchronize()
        bwd_times.append(start_bwd.elapsed_time(end_bwd))
        
    avg_fwd = np.mean(fwd_times)
    avg_bwd = np.mean(bwd_times)
    
    return avg_fwd, avg_bwd, fwd_mem, bwd_mem

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("Benchmarking requires a CUDA GPU.")

    # Standard ViT parameters
    BATCH_SIZE = 32
    SEQ_LENGTH = 197
    D_HIDDEN = 768
    NUM_HEADS = 12
    K = 8
    
    print(f"Benchmarking with Batch Size: {BATCH_SIZE}, Seq Length: {SEQ_LENGTH}")
    print("-" * 60)

    # Initialize Modules
    old_model = MultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH
    ).to(device)
    
    new_model = FastMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH
    ).to(device)

    # We reuse the exact same input tensor to ensure fairness
    x = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device, requires_grad=True)

    print("Benchmarking Original Implementation...")
    old_fwd, old_bwd, old_fmem, old_bmem = benchmark_module(old_model, x)

    print("Benchmarking Triton Implementation...")
    new_fwd, new_bwd, new_fmem, new_bmem = benchmark_module(new_model, x)

    # Print Results Table
    print("\n" + "=" * 60)
    print(f"{'Metric':<25} | {'Original':<12} | {'Triton':<12} | {'Improvement'}")
    print("-" * 60)
    print(f"{'Forward Time (ms)':<25} | {old_fwd:<12.2f} | {new_fwd:<12.2f} | {old_fwd/new_fwd:.2f}x faster")
    print(f"{'Backward Time (ms)':<25} | {old_bwd:<12.2f} | {new_bwd:<12.2f} | {old_bwd/new_bwd:.2f}x faster")
    print(f"{'Forward Peak Memory (MB)':<25} | {old_fmem:<12.2f} | {new_fmem:<12.2f} | {old_fmem/new_fmem:.2f}x less")
    print(f"{'Backward Peak Memory (MB)':<25} | {old_bmem:<12.2f} | {new_bmem:<12.2f} | {old_bmem/new_bmem:.2f}x less")
    print("=" * 60)

Benchmarking with Batch Size: 32, Seq Length: 197
------------------------------------------------------------
Benchmarking Original Implementation...
Benchmarking Triton Implementation...

Metric                    | Original     | Triton       | Improvement
------------------------------------------------------------
Forward Time (ms)         | 4.92         | 4.24         | 1.16x faster
Backward Time (ms)        | 9.29         | 6.84         | 1.36x faster
Forward Peak Memory (MB)  | 795.01       | 341.95       | 2.32x less
Backward Peak Memory (MB) | 904.88       | 657.54       | 1.38x less


In [12]:
import torch

# (Assume MultiHeadConvNNAttention and FastMultiHeadConvNNAttention are defined above)

def verify_correctness():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("Validation requires a CUDA GPU.")

    # Hyperparameters
    BATCH_SIZE = 4
    SEQ_LENGTH = 197
    D_HIDDEN = 768
    NUM_HEADS = 12
    K = 8
    
    # 1. Initialize models
    # CRITICAL: Dropout must be 0.0 so both models are deterministic!
    torch.manual_seed(42)
    old_model = MultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, convolution_type='depthwise', seq_length=SEQ_LENGTH
    ).to(device)
    
    new_model = FastMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, seq_length=SEQ_LENGTH
    ).to(device)

    # 2. Synchronize Weights
    print("Copying weights from Original to Triton model...")
    with torch.no_grad():
        new_model.W_q.weight.copy_(old_model.W_q.weight)
        new_model.W_k.weight.copy_(old_model.W_k.weight)
        new_model.W_v.weight.copy_(old_model.W_v.weight)
        new_model.W_o.weight.copy_(old_model.W_o.weight)
        # old_model.conv is an nn.Conv1d for depthwise, its weight shape is (d_k, 1, K)
        new_model.conv_weight.copy_(old_model.conv.weight)

    # 3. Create identical inputs (cloned so they track gradients separately)
    x_base = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device)
    x_old = x_base.clone().requires_grad_(True)
    x_new = x_base.clone().requires_grad_(True)

    # ==========================================
    # FORWARD PASS CHECK
    # ==========================================
    print("\nRunning Forward Passes...")
    out_old = old_model(x_old)
    out_new = new_model(x_new)

    fwd_diff = (out_old - out_new).abs().max().item()
    fwd_match = torch.allclose(out_old, out_new, atol=1e-4)
    
    print(f"Forward Pass Match: {fwd_match}")
    print(f"Max Forward Difference: {fwd_diff:.8f}")

    # ==========================================
    # BACKWARD PASS CHECK
    # ==========================================
    print("\nRunning Backward Passes...")
    # Generate a random incoming gradient from the "next layer"
    grad_out = torch.randn_like(out_old)

    out_old.backward(grad_out)
    out_new.backward(grad_out)

    # Check Input Gradients
    x_grad_diff = (x_old.grad - x_new.grad).abs().max().item()
    x_grad_match = torch.allclose(x_old.grad, x_new.grad, atol=1e-4)
    print(f"Input Gradient (x.grad) Match: {x_grad_match}")
    print(f"Max Input Gradient Difference: {x_grad_diff:.8f}")

    # Check Weight Gradients (Convolution)
    conv_grad_diff = (old_model.conv.weight.grad - new_model.conv_weight.grad).abs().max().item()
    conv_grad_match = torch.allclose(old_model.conv.weight.grad, new_model.conv_weight.grad, atol=1e-4)
    print(f"Conv Weight Gradient Match: {conv_grad_match}")
    print(f"Max Conv Weight Gradient Difference: {conv_grad_diff:.8f}")

    # Check Weight Gradients (Linear Projections - Example: W_q)
    wq_grad_diff = (old_model.W_q.weight.grad - new_model.W_q.weight.grad).abs().max().item()
    wq_grad_match = torch.allclose(old_model.W_q.weight.grad, new_model.W_q.weight.grad, atol=1e-4)
    print(f"W_q Weight Gradient Match: {wq_grad_match}")
    print(f"Max W_q Weight Gradient Difference: {wq_grad_diff:.8f}")

    print("\n" + "="*50)
    if fwd_match and x_grad_match and conv_grad_match and wq_grad_match:
        print("SUCCESS: Triton implementation is mathematically equivalent!")
    else:
        print("WARNING: Divergence detected between implementations.")
    print("="*50)

if __name__ == "__main__":
    verify_correctness()

Copying weights from Original to Triton model...

Running Forward Passes...
Forward Pass Match: True
Max Forward Difference: 0.00000000

Running Backward Passes...
Input Gradient (x.grad) Match: True
Max Input Gradient Difference: 0.00000015
Conv Weight Gradient Match: True
Max Conv Weight Gradient Difference: 0.00000286
W_q Weight Gradient Match: True
Max W_q Weight Gradient Difference: 0.00000262

SUCCESS: Triton implementation is mathematically equivalent!
